In [2]:
from multiprocessing import set_start_method
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# **Must** happen before torch or vllm ever touches CUDA
set_start_method("spawn", force=True)

from datasets import interleave_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast, PreTrainedModel
from transformers import Trainer, TrainingArguments
from trl import SFTConfig, SFTTrainer
from trl import setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from trl import DataCollatorForCompletionOnlyLM
import torch
from vllm import LLM, SamplingParams
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from drgrpo_grader import r1_zero_reward_fn
import gc
from unittest.mock import patch
import wandb
import safetensors
import os
import json
import numpy as np
import time
import random 
import operator
import itertools
from transformers import TrainerCallback, TrainerState, TrainerControl


INFO 06-26 23:17:47 [__init__.py:247] No platform detected, vLLM is running on UnspecifiedPlatform
WARNING 06-26 23:17:48 [_custom_ops.py:21] Failed to import from vllm._C with ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


In [5]:
def compute_group_normalized_rewards(
    reward_fn,
    rollout_responses,
    repeated_ground_truths,
    group_size,
    advantage_eps,
    normalize_by_std,
    ):
    """
    Compute rewards for each group of rollout responses, normalized by the group size.
    Args:
    reward_fn: Callable[[str, str], dict[str, float]] Scores the rollout responses against
    the ground truths, producing a dict with keys "reward", "format_reward", and
    "answer_reward".
    rollout_responses: list[str] Rollouts from the policy. The length of this list is
    rollout_batch_size = n_prompts_per_rollout_batch * group_size.
    repeated_ground_truths: list[str] The ground truths for the examples. The length of this
    list is rollout_batch_size, because the ground truth for each example is repeated
    group_size times.
    group_size: int Number of responses per question (group).
    advantage_eps: float Small constant to avoid division by zero in normalization.
    normalize_by_std: bool If True, divide by the per-group standard deviation; otherwise
    subtract only the group mean.
    Returns:
    tuple[torch.Tensor, torch.Tensor, dict[str, float]].
    advantages shape (rollout_batch_size,). Group-normalized rewards for each rollout
    response.
    raw_rewards shape (rollout_batch_size,). Unnormalized rewards for each rollout
    response.
    metadata your choice of other statistics to log (e.g. mean, std, max/min of rewards).
    """
    rollout_batch_size = len(rollout_responses)
    n_prompts_per_rollout_batch = rollout_batch_size // group_size
    # ground_truths = [[gt]*group_size for gt in repeated_ground_truths]
    # ground_truths = list(itertools.chain.from_iterable(ground_truths))
    rewards = []
   
    for response, gt in zip(rollout_responses, repeated_ground_truths):
        reward = reward_fn(response, gt)
        rewards.append(reward['reward'])
    raw_rewards = torch.Tensor(rewards)
   
    rewards = raw_rewards.view(n_prompts_per_rollout_batch, group_size)
    mean_rewards =  torch.mean(rewards, dim=1, keepdim=True)
    std_rewards = torch.std(rewards, dim=1, keepdim=True)
    advantages = rewards - mean_rewards
    if normalize_by_std:
        advantages = advantages / (std_rewards + advantage_eps)
    advantages = advantages.view(-1)

    metadata = dict()
    metadata['std_rewards'] = torch.std(raw_rewards).item()
    metadata['mean_rewards'] = torch.mean(raw_rewards).item()
    metadata['max_rewards'] = torch.max(raw_rewards).item()
    metadata['min_rewards'] = torch.min(raw_rewards).item()
    return advantages, raw_rewards, metadata


def compute_naive_policy_gradient_loss(
    raw_rewards_or_advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    ) -> torch.Tensor:
    """
    Compute the policy-gradient loss at every token, where raw_rewards_or_advantages is either
    the raw reward or an already-normalized advantage.
    Args:
    raw_rewards_or_advantages: torch.Tensor Shape (batch_size, 1), scalar
    reward/advantage for each rollout response.
    policy_log_probs: torch.Tensor Shape (batch_size, sequence_length), logprobs for
    each token.
    Returns:
    torch.Tensor Shape (batch_size, sequence_length), the per-token policy-gradient loss (to
    be aggregated across the batch and sequence dimensions in the training loop).
    """
    return raw_rewards_or_advantages * policy_log_probs

def compute_grpo_clip_loss(
    advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    cliprange: float,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    Args:
    advantages: torch.Tensor Shape (batch_size, 1), per-example advantages A.
    policy_log_probs: torch.Tensor Shape (batch_size, sequence_length), per-token log
    probs from the policy being trained.
    old_log_probs: torch.Tensor Shape (batch_size, sequence_length), per-token log probs
    from the old policy.
    cliprange: float Clip parameter ϵ (e.g. 0.2).
    Returns:
    tuple[torch.Tensor, dict[str, torch.Tensor]].
    loss torch.Tensor of shape (batch_size, sequence_length), the per-token clipped
    loss.
    metadata dict containing whatever you want to log. We suggest logging whether each
    token was clipped or not, i.e., whether the clipped"""
    ratio = torch.exp(policy_log_probs - old_log_probs)

    surrogant1 = advantages * ratio

    clipped = torch.clamp(ratio, 1 - cliprange, 1 + cliprange)
    surrogant2 = advantages * clipped
    per_token_loss = - torch.min(surrogant1, surrogant2)
    metadata = dict()
    clipped_mask = (clipped != ratio)
    metadata['clip_mask'] = clipped_mask
    

    return per_token_loss, metadata


def compute_policy_gradient_loss(
    policy_log_probs: torch.Tensor,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
    ) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """
    Select and compute the desired policy-gradient loss.
    Args:
    policy_log_probs (batch_size, sequence_length), per-token log-probabilities from the
    policy being trained.
    loss_type One of "no_baseline", "reinforce_with_baseline", or "grpo_clip".
    raw_rewards Required if loss_type == "no_baseline"; shape (batch_size, 1).
    advantages Required for "reinforce_with_baseline" and "grpo_clip"; shape
    (batch_size, 1).
    old_log_probs Required for "grpo_clip"; shape (batch_size, sequence_length).
    cliprange Required for "grpo_clip"; scalar ϵ used for clipping.
    Returns:
    tuple[torch.Tensor, dict[str, torch.Tensor]].
    loss (batch_size, sequence_length), per-token loss.
    metadata dict, statistics from the underlying routine (e.g., clip fraction for GRPO-Clip)."""
    metadata_final = dict()
    if loss_type == 'no_baseline':
        assert raw_rewards is not None
        loss = compute_naive_policy_gradient_loss(raw_rewards,  policy_log_probs)
    if loss_type == 'reinforce_with_baseline':
        assert advantages is not None
        loss = compute_naive_policy_gradient_loss(advantages,  policy_log_probs)
    if loss_type == 'grpo_clip':
        assert advantages is not None
        assert old_log_probs is not None
        assert cliprange is not None
        loss, metadata = compute_grpo_clip_loss(
            advantages,
            policy_log_probs,
            old_log_probs, 
            cliprange,
        ) 
        metadata_final.update(metadata)
    return loss, metadata_final
    





SyntaxError: incomplete input (1069838208.py, line 137)

In [4]:

ground_truths = [[gt]*2 for gt in ['1','b']]
ground_truths = list(itertools.chain.from_iterable(ground_truths))
ground_truths

['1', '1', 'b', 'b']